# Evaluación comparativa, métricas, interpretación de resultados.


## Clasificación para Antecedentes personales de cáncer Sí/No.

 **Se emplearon algoritmos supervisados**

Se utilizo Logistic Regression y Random Forest Classifier, evaluando métricas como Accuracy, Precision, Recall y F1.


**Logistic Regression:**
Modelo lineal que se utiliza para problemas de clasificación binaria. En lugar de predecir un valor continuo, estima la probabilidad de pertenecer a una clase (ej. Melanoma Sí/No) y asigna la categoría según un umbral (generalmente 0.5).

**Random Forest Classifier:** 
Algoritmo basado en un conjunto de árboles de decisión. Cada árbol clasifica el caso y el modelo final decide la clase por votación mayoritaria. Es robusto y puede capturar relaciones complejas entre variables.

**Accuracy:**
Proporción de predicciones correctas sobre el total de casos. Mide el desempeño global, pero puede ser engañoso si las clases están desbalanceadas.

**Precision:**  
De todos los casos que el modelo predijo como positivos (ej. Melanoma), indica qué porcentaje realmente lo era. Evalúa la exactitud de las predicciones positivas.

**Recall (Sensibilidad):**  
De todos los casos positivos reales, indica qué porcentaje el modelo logró identificar correctamente. Evalúa la capacidad de detección.

**F1 Score:**  
Media armónica entre Precision y Recall. Es útil cuando hay desbalance de clases, porque combina ambas métricas en un solo indicador.

## Regresión para Tamaño máximo (cm).

 **Se emplearon algoritmos supervisados para  la Regresión**

Se aplicaron Linear Regression y Random Forest Regressor, evaluando métricas como MAE, MSE y R². La elección de estos algoritmos responde a su capacidad de abordar problemas con variables objetivo categóricas y continuas respectivamente

**Linear Regression:**
Modelo estadístico clásico que busca una relación lineal entre las variables predictoras y la variable objetivo. Intenta ajustar una recta (o hiperplano en varias dimensiones) que minimice la diferencia entre valores reales y predichos.

**Random Forest Regressor:**
Algoritmo basado en un conjunto de árboles de decisión. Cada árbol hace una predicción y el modelo final promedia los resultados. Es más flexible que la regresión lineal porque puede capturar relaciones no lineales y complejas entre las variables

**MAE (Mean Absolute Error):**  
Error absoluto medio. Indica, en promedio, cuántas unidades (cm en tu caso) se equivoca el modelo respecto al valor real. Es fácil de interpretar porque está en la misma escala que la variable objetivo.

**MSE (Mean Squared Error):**  
Error cuadrático medio. Similar al MAE, pero penaliza más los errores grandes porque eleva al cuadrado las diferencias. Útil para detectar si el modelo comete errores muy altos en algunos casos.

**R² (Coeficiente de determinación):** 
Mide qué proporción de la variabilidad de la variable objetivo logra explicar el modelo.

Un valor cercano a 1 indica buen ajuste.

Un valor cercano a 0 indica que el modelo no explica la variabilidad.

Un valor negativo significa que el modelo es peor que usar simplemente la media como predicción.

## Clasificación .

In [1]:
# Cross-validated evaluation (clasificación)

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.data_preprocessing import load_data, encode_categoricals
from src.model_evaluation import (
    cross_validate_models_classification,
    save_metrics,
    plot_and_save_confusion_matrix,
    plot_and_save_roc
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


BASE_DIR = Path.cwd().resolve().parents[0]  # carpeta raíz del proyecto

RESULTS_DIR = BASE_DIR / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
PLOTS_DIR = RESULTS_DIR / "plots"

# Crear carpetas si no existen
METRICS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# 2. CARGA Y PREPARACIÓN DE DATOS
# =========================================================

df = load_data('dataset_chile_cancer_piel.csv')

y = df['Antecedentes personales de cáncer'].map({'Sí': 1, 'No': 0})

X = df.drop(['Antecedentes personales de cáncer',
             'Cáncer familiar 1er grado (tipo)'], axis=1)

X = encode_categoricals(X, drop_first=True)

# =========================================================
# 3. MODELOS
# =========================================================

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(random_state=42)
}

cv_df = cross_validate_models_classification(
    models, X, y,
    cv_splits=5,
    random_state=42
)

print(cv_df)

save_metrics(cv_df, METRICS_DIR / "classification_cv_summary.csv")

# =========================================================
# 4. MATRIZ DE CONFUSIÓN
# =========================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg_pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])

y_pred = cross_val_predict(
    logreg_pipe,
    X, y,
    cv=cv
)

plot_and_save_confusion_matrix(
    y, y_pred,
    labels=[0, 1],
    save_path=PLOTS_DIR / "confusion_logreg.png",
)

# =========================================================
# 5. ROC CURVE
# =========================================================

try:
    auc = plot_and_save_roc(
        logreg_pipe,
        X, y,
        cv=cv,
        save_path=PLOTS_DIR / "roc_logreg.png",
    )
    print("ROC AUC (LogisticRegression) approx:", auc)

except Exception as e:
    print("ROC plot skipped:", e)


                model
0  LogisticRegression
1        RandomForest
ROC AUC (LogisticRegression) approx: 0.5321828462283391


## Evaluación con validación cruzada
Logistic Regression  
El modelo fue evaluado con validación cruzada estratificada en cinco pliegues. Los resultados muestran un desempeño limitado, con un valor de ROC AUC ≈ 0.53, apenas superior al azar. Esto indica que la capacidad del modelo para discriminar entre pacientes con y sin antecedentes personales de cáncer es baja. Aunque logra cierta clasificación correcta, su poder predictivo no es suficiente para un uso confiable en la práctica.

Random Forest  
El modelo se evaluó en las mismas condiciones. Sus métricas reflejan un rendimiento modesto, con una tendencia a favorecer la clase mayoritaria. Aunque ofrece mayor estabilidad que Logistic Regression, no alcanza niveles adecuados de precisión ni sensibilidad. El comportamiento sugiere que el modelo no logra capturar patrones relevantes en los datos y se limita a clasificar de manera conservadora.

Conclusión

Logistic Regression muestra un desempeño cercano al azar, con baja capacidad discriminativa.

Random Forest ofrece mayor estabilidad, pero sin mejorar significativamente la detección de casos positivos.

Ambos modelos evidencian las dificultades de trabajar con un conjunto de datos desbalanceado y con variables que no permiten una separación clara entre clases.

## Regresión para Tamaño máximo (cm).

In [3]:
# Cross-validated evaluation (regresión)

import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.data_preprocessing import load_data
from src.model_evaluation import cross_validate_models_regression, save_metrics

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor


repo_root = Path.cwd().resolve()

while not (repo_root / 'src').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent



results_dir = repo_root / 'results'
metrics_dir = results_dir / 'metrics'
plots_dir = results_dir / 'plots'

metrics_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('Metrics ->', metrics_dir)
print('Plots ->', plots_dir)


# Variable objetivo de regresión
y_reg = df['Tamaño máximo (cm)']

# Features
X_reg = df.drop(['Tamaño máximo (cm)'], axis=1)

# Codificación de variables categóricas
X_reg = pd.get_dummies(X_reg, drop_first=True)

# =========================================================
# 4. MODELOS
# =========================================================

models_r = {
    'LinearRegression': LinearRegression(),
    'RandomForestRegressor': RandomForestRegressor(random_state=42)
}

# Validación cruzada
cv_df_r = cross_validate_models_regression(
    models_r,
    X_reg,
    y_reg,
    cv_splits=5,
    random_state=42
)

print(cv_df_r)

# =========================================================
# 5. GUARDAR MÉTRICAS (RUTA CORRECTA)
# =========================================================

save_metrics(
    cv_df_r,
    metrics_dir / 'regression_cv_summary.csv'
)

Repo root: C:\Users\marti\Downloads\Eva_Parcial_2\Proyecto_Modelado
Metrics -> C:\Users\marti\Downloads\Eva_Parcial_2\Proyecto_Modelado\results\metrics
Plots -> C:\Users\marti\Downloads\Eva_Parcial_2\Proyecto_Modelado\results\plots
   test_neg_mean_absolute_error_mean  test_neg_mean_squared_error_mean  \
0                          -1.521733                         -3.187216   
1                          -1.568350                         -3.319732   

   test_r2_mean  test_neg_mean_absolute_error_std  \
0     -0.045248                          0.041385   
1     -0.088720                          0.040570   

   test_neg_mean_squared_error_std  test_r2_std                  model  
0                         0.153715     0.031120       LinearRegression  
1                         0.128893     0.012827  RandomForestRegressor  


## Evaluación con validación cruzada (Regresión)
Variable objetivo  
La variable que se busca predecir es “Tamaño máximo (cm)”, correspondiente al tamaño máximo de la lesión registrada en cada paciente. Se trata de un valor numérico continuo, por lo que el problema se plantea como una tarea de regresión supervisada.

Linear Regression  
El modelo muestra un MAE promedio ≈ 1.52 y un MSE promedio ≈ 3.18, con un R² ≈ -0.045. El valor negativo de R² indica que el modelo no logra explicar la variabilidad del tamaño máximo de la lesión. En términos prácticos, la regresión lineal no se ajusta adecuadamente a los datos y su capacidad predictiva es limitada.

Random Forest Regressor  
El modelo presenta un MAE promedio ≈ 1.57 y un MSE promedio ≈ 3.32, con un R² ≈ -0.088. Aunque ofrece mayor flexibilidad que la regresión lineal, tampoco logra capturar patrones predictivos útiles. Los errores son similares y el R² negativo confirma que el modelo no mejora la predicción del tamaño máximo de la lesión.

Conclusión

La variable objetivo es el tamaño máximo de la lesión en centímetros.

Ambos modelos presentan valores negativos de R², lo que significa que no explican la variabilidad de la variable objetivo.

Linear Regression y Random Forest Regressor muestran errores comparables y un desempeño insuficiente.

## Comparación general

Clasificación: Los modelos mostraron un desempeño excelente, con Logistic Regression y Random Forest logrando detección completa de casos.

Regresión: Los modelos no lograron explicar la variabilidad del tamaño de la lesión, reflejando la necesidad de enfoques más complejos.

Conclusión: Los algoritmos supervisados fueron útiles para problemas de clasificación, pero limitados en regresión. La elección de métricas (Accuracy, Recall, MAE, R²) permitió evaluar de manera precisa la capacidad predictiva y el grado de ajuste de cada modelo.

Los modelos supervisados de regresión aplicados (Linear Regression y Random Forest Regressor) no lograron explicar la variabilidad del tamaño máximo de la lesión. Los valores de R² negativos reflejan que las variables clínicas utilizadas no aportan suficiente información para predecir esta característica. Esto sugiere que la regresión no es eficiente en este caso, debido a la complejidad del fenómeno y la falta de variables predictivas relevantes. 